# Plasmid size and copy number: an explorer

Interactive companion to [`RESULTS.md`](https://github.com/Azuromalachite/plasmid-ecoevo-lecture-2026/blob/main/reproductions/ramiro-scaling/RESULTS.md) and [`REPORT.md`](https://github.com/Azuromalachite/plasmid-ecoevo-lecture-2026/blob/main/reproductions/ramiro-scaling/REPORT.md).

**What this is.** One dataset, drawn two ways, with the filters live. The two published panels this imitates are:

| panel | paper | what it plots |
|---|---|---|
| A | Shahzadi et al., *Nat Commun* (2026) | copy number against plasmid length, coloured by chromosome length |
| B | Maddamsetti et al., *Nat Commun* **16**:6023 (2025) | copy number against chromosome-normalised length, coloured by mobility, with a bend |

**What this is NOT.** It does not reproduce either figure. Both are drawn here from a third dataset — the curated modelling subset released by Ramiro-Martínez et al. (2025) — because that is the one we can fetch and verify by hash. Shapes that look like the published panels are the same *relationship* measured on a different sample, not the same measurement. Section 5 at the end says exactly where the published data lives and why this notebook cannot download it for you.

**Copy number here is modelled, not counted** — a sequencing-coverage ratio — and the source ships it as a point estimate with no uncertainty of any kind. Nothing below should be read as resolving a difference smaller than an interval we cannot see.


In [ ]:
# @title Fetch the dataset and check its hash
import hashlib, io, urllib.request
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

URL = ("https://raw.githubusercontent.com/PaulaRamiro/NpAUREO/main/Model/curateddataset.csv")
EXPECTED_SHA256 = "23bee155b1c946bd565840c4cc0f5ac2e1fec6e4192578ba9f6752ee5f58f78f"

req = urllib.request.Request(URL, headers={"User-Agent": "plasmid-ecoevo-repro/1.0"})
raw = urllib.request.urlopen(req, timeout=120).read()
digest = hashlib.sha256(raw).hexdigest()

print(f"bytes   {len(raw):,}")
print(f"sha256  {digest}")
if digest == EXPECTED_SHA256:
    print("        matches the bytes the analysis was written against")
else:
    print("        DIFFERS from the recorded input. `main` moves. Every")
    print("        number quoted in the markdown below may no longer")
    print("        hold; re-run reproduce.py to get a fresh receipt.")


In [ ]:
# @title Load and derive
df = pd.read_csv(io.BytesIO(raw), sep="\t", encoding="utf-8-sig")
df = df[(df.PCN > 0) & (df.Length_chr > 0)].copy()

df["plasmid_bp"] = 10 ** df.Logsize
df["rel_pct"] = 100 * df.plasmid_bp / df.Length_chr   # % of chromosome
df["x"] = np.log10(df.rel_pct)
df["y"] = np.log10(df.PCN)

print(f"{len(df):,} plasmids, {df.Predicted_ptu.nunique()} PTUs, "
      f"{df.Genus.nunique()} genera")
print(f"PCN   {df.PCN.min():.3f} to {df.PCN.max():,.0f}")
print(f"size  {df.plasmid_bp.min():,.0f} to {df.plasmid_bp.max():,.0f} bp")
print()
print("NOTE: minimum PCN is exactly 1.000 -- the authors excluded")
print("anything below, which truncates the quantity being fitted.")
print("Every record also carries a named PTU, so unclassifiable")
print("plasmids were dropped upstream too. Neither is stated in the file.")


## 1. The fitting machinery

A continuous two-segment fit, scanned over candidate knee positions. `dAIC = AIC(segmented) - AIC(single line)`, so **negative favours the bend**; the single line is fitted with two parameters against the hinge's four.


In [ ]:
# @title Segmented fit and model comparison
def ols(x, y):
    k, b = np.polyfit(x, y, 1)
    sse = float(((y - (k * x + b)) ** 2).sum())
    return k, b, sse

def hinge(x, y, c):
    M = np.column_stack([np.ones_like(x), x, np.maximum(x - c, 0.0)])
    beta, *_ = np.linalg.lstsq(M, y, rcond=None)
    return float(((y - M @ beta) ** 2).sum()), beta

# n_grid matches reproduce.py exactly. A coarser scan here moves the
# knee by 0.01 percentage points and the upper slope in the third
# decimal, which would leave this notebook quietly disagreeing with the
# receipt it is meant to illustrate.
def best_hinge(x, y, n_grid=400):
    grid = np.linspace(np.quantile(x, 0.05), np.quantile(x, 0.95), n_grid)
    sse, c = min((hinge(x, y, g)[0], g) for g in grid)
    _, beta = hinge(x, y, c)
    return c, float(beta[1]), float(beta[1] + beta[2]), sse

def aic(sse, n, k):
    return n * np.log(sse / n) + 2 * k

def compare(x, y):
    """Returns (knee, slope_lo, slope_hi, dAIC). Negative dAIC favours the bend."""
    _, _, sse1 = ols(x, y)
    c, lo, hi, sse2 = best_hinge(x, y)
    n = len(x)
    return c, lo, hi, aic(sse2, n, 4) - aic(sse1, n, 2)


## 2. Panel A — the form of the Shahzadi figure

Copy number against plasmid length, points coloured by the length of the host chromosome. The published panel reports an exponent near **−0.71**; what this sample gives is printed under the plot. They are different samples, so a difference between the two exponents is not a disagreement about anything until someone matches the accessions.


In [ ]:
# @title Panel A
fig, ax = plt.subplots(figsize=(6.4, 4.6), dpi=120)
sc = ax.scatter(df.Logsize, df.y, c=df.Length_chr, s=6, alpha=0.5,
                cmap="coolwarm", norm=Normalize(*np.quantile(df.Length_chr, [0.02, 0.98])),
                linewidths=0)
cb = fig.colorbar(sc, ax=ax); cb.set_label("chromosome length (bp)")

k, b, _ = ols(df.Logsize.values, df.y.values)
xs = np.linspace(df.Logsize.min(), df.Logsize.max(), 100)
ax.plot(xs, k * xs + b, color="#000000", lw=1.6)
ax.set_xlabel("plasmid length, log10 bp")
ax.set_ylabel("copy number, log10")
ax.set_title(f"PCN $\\propto$ size$^{{{k:.2f}}}$   (Ramiro subset, n={len(df):,})",
             fontsize=11)
ax.axhline(0, color="#888888", lw=0.8, ls="-")
plt.tight_layout(); plt.show()

print(f"this sample     exponent {k:+.3f}")
print(f"Shahzadi 2026   exponent -0.71  (different sample, not comparable row-by-row)")


## 3. Panel B — the form of the Maddamsetti figure

The same plasmids on the chromosome-normalised axis, coloured by the authors' predicted mobility class, with the two-segment fit drawn on top. Maddamsetti et al. put their bend near **1.84%** of chromosome length. The dashed line marks that value so you can see where it falls on *this* sample; the solid bend is what this sample actually fits.


In [ ]:
# @title Panel B
MOB_COLOUR = {"conjugative": "#E69F00", "mobilizable": "#009E73",
              "non-mobilizable": "#56B4E9"}

fig, ax = plt.subplots(figsize=(6.6, 4.6), dpi=120)
for mob, g in df.groupby("predicted_mobility"):
    ax.scatter(g.x, g.y, s=6, alpha=0.55, linewidths=0,
               c=MOB_COLOUR.get(str(mob).strip(), "#999999"), label=f"{mob} (n={len(g):,})")

c, lo, hi, d = compare(df.x.values, df.y.values)
xs = np.linspace(df.x.min(), df.x.max(), 200)
_, beta = hinge(df.x.values, df.y.values, c)   # the fitted hinge itself,
# not a reconstruction: anchoring a two-slope line by hand is how a drawn
# curve comes to disagree with the model that was actually compared.
ax.plot(xs, beta[0] + beta[1] * xs + beta[2] * np.maximum(xs - c, 0),
        color="#C1272D", lw=2, zorder=5)
ax.axvline(np.log10(1.84), color="#C1272D", lw=1.1, ls="--", zorder=4)
ax.text(np.log10(1.84), ax.get_ylim()[1], "  1.84% (Maddamsetti)",
        va="top", fontsize=8, color="#C1272D")
ax.axhline(0, color="#888888", lw=0.8)
ax.set_xlabel("plasmid length as % of chromosome, log10")
ax.set_ylabel("copy number, log10")
ax.legend(fontsize=8, frameon=False, loc="lower left")
plt.tight_layout(); plt.show()

print(f"this sample   bend at {10**c:.2f}% of chromosome, slopes {lo:+.3f} then {hi:+.3f}")
print(f"              dAIC {d:+.1f}  ({'segmented' if d < 0 else 'single line'} preferred)")
print(f"Maddamsetti   bend at 1.84%, slopes -0.880 then -0.125")


## 4. The explorer

Move the controls and watch the bend. The thing worth doing first: set **centre on PTU** and watch dAIC collapse. That is the finding in `REPORT.md` — the bend is a between-family effect, and removing each family's own mean removes it.

Second thing worth doing: raise the **PCN floor**. The exponent flattens monotonically, because the floor is a one-sided truncation that falls almost entirely on large plasmids.


In [ ]:
# @title Interactive explorer
import ipywidgets as W
from IPython.display import display

genera = ["(all)"] + sorted(df.Genus.dropna().unique().tolist())
mobs = ["(all)"] + sorted(df.predicted_mobility.dropna().astype(str).unique().tolist())

w_genus = W.Dropdown(options=genera, value="(all)", description="genus")
w_mob = W.Dropdown(options=mobs, value="(all)", description="mobility")
w_floor = W.FloatSlider(value=1.0, min=1.0, max=10.0, step=0.5,
                        description="PCN floor", continuous_update=False)
w_minptu = W.IntSlider(value=1, min=1, max=50, step=1,
                       description="min n per PTU", continuous_update=False)
w_centre = W.Checkbox(value=False, description="centre on PTU")

def explore(genus, mob, floor, min_ptu, centre):
    d = df
    if genus != "(all)": d = d[d.Genus == genus]
    if mob != "(all)": d = d[d.predicted_mobility.astype(str) == mob]
    d = d[d.PCN >= floor]
    if min_ptu > 1:
        keep = d.Predicted_ptu.value_counts()
        d = d[d.Predicted_ptu.isin(keep[keep >= min_ptu].index)]
    if len(d) < 40:
        print(f"only {len(d)} plasmids left -- too few to fit"); return

    x, y = d.x.values, d.y.values
    label = "raw"
    if centre:
        g = d.groupby("Predicted_ptu")
        big = g.x.transform("count") >= 3
        d2 = d[big]
        x = (d2.x - d2.groupby("Predicted_ptu").x.transform("mean")).values
        y = (d2.y - d2.groupby("Predicted_ptu").y.transform("mean")).values
        label = f"centred on {d2.Predicted_ptu.nunique()} PTUs"

    c, lo, hi, dd = compare(x, y)
    k, b, _ = ols(x, y)

    fig, ax = plt.subplots(figsize=(6.6, 4.2), dpi=110)
    ax.scatter(x, y, s=6, alpha=0.45, linewidths=0, c="#56B4E9")
    xs = np.linspace(x.min(), x.max(), 200)
    ax.plot(xs, k * xs + b, color="#888888", lw=1.2, ls="--", label=f"single line {k:+.3f}")
    _, beta = hinge(x, y, c)
    ax.plot(xs, beta[0] + beta[1] * xs + beta[2] * np.maximum(xs - c, 0),
            color="#C1272D", lw=2, label=f"segmented {lo:+.3f} / {hi:+.3f}")
    ax.set_xlabel("log10 % of chromosome" + (" (family-centred)" if centre else ""))
    ax.set_ylabel("log10 copy number" + (" (family-centred)" if centre else ""))
    ax.set_title(f"n={len(x):,}  {label}   dAIC {dd:+.1f}", fontsize=11)
    ax.legend(fontsize=8, frameon=False)
    plt.tight_layout(); plt.show()

    verdict = "segmented preferred" if dd < -4 else ("no clear preference" if dd > -2 else "marginal")
    print(f"dAIC {dd:+8.1f}   {verdict}")
    if not centre and dd < -4:
        print("   now tick 'centre on PTU' and watch this number")

display(W.interactive(explore, genus=w_genus, mob=w_mob, floor=w_floor,
                      min_ptu=w_minptu, centre=w_centre))


## 5. The published datasets, and why this notebook cannot fetch them

Checked 6 November 2026.

**Maddamsetti et al. 2025** ([10.1038/s41467-025-61205-2](https://doi.org/10.1038/s41467-025-61205-2), PMC12222811) states: *"All processed data in this study are provided in the Source Data provided with this paper."* Their code is public:

| repository | contents | Zenodo |
|---|---|---|
| [`pseuPIRA`](https://github.com/rohanmaddamsetti/pseuPIRA) | the copy-number estimator | `10.5281/zenodo.15668298` |
| [`PCN-db-pipeline`](https://github.com/rohanmaddamsetti/PCN-db-pipeline) | the estimation pipeline | `10.5281/zenodo.15668288` |
| [`plasmid-scaling-laws`](https://github.com/rohanmaddamsetti/plasmid-scaling-laws) | statistics and figures (`PCN-analysis.R`) | `10.5281/zenodo.15668292` |

What is **not** in those repositories is the per-plasmid copy-number table. `PCN-db-pipeline/results/complete-prokaryotes-with-plasmids.txt` is the NCBI genome manifest — accessions, replicon lengths, assembly IDs, no copy numbers. The copy numbers are in the article's Source Data, and PMC serves supplementary files behind a "Preparing to download" interstitial that an automated request cannot pass. So this notebook does not try.

**To add their data yourself:** download the Source Data from the article page, upload the CSV to this Colab session, and run the cell below. It will tell you which columns it found rather than guessing at a schema it has not seen.

**Shahzadi et al. 2026** is from the same group (*Nat Commun* 17:5539). Treat any agreement between it and Maddamsetti as shared provenance until the accessions are matched, not as independent replication.


In [ ]:
# @title Optional: load Maddamsetti Source Data if you have it
# Upload the Source Data CSV in the Colab file pane, then set the name.
MADDAMSETTI_CSV = ""  # e.g. "41467_2025_61205_MOESM8_ESM.csv"

if not MADDAMSETTI_CSV:
    print("No file set. See the markdown above for where to get it.")
else:
    import os
    if not os.path.exists(MADDAMSETTI_CSV):
        print(f"not found: {MADDAMSETTI_CSV}")
    else:
        m = pd.read_csv(MADDAMSETTI_CSV)
        print(f"{len(m):,} rows, {len(m.columns)} columns")
        for col in m.columns:
            print(f"  {col}")
        print()
        print("Before plotting anything from this: check which column is")
        print("copy number, which is plasmid length, and whether length is")
        print("already normalised by chromosome. Do not assume the schema")
        print("matches the Ramiro file -- it is a different pipeline.")


---

### What this notebook does not establish

- It reproduces no published figure. Panels A and B imitate the *form* of two published panels using a third dataset.
- Copy number is modelled with no published uncertainty, so no difference smaller than an unknown interval means anything here.
- The dataset carries no accession, so nothing in it can be joined to either published dataset, to PTU membership tables, or to replicon typing. The interesting comparisons need the accessions.
- Plasmids within a PTU are related by descent and are not independent observations; the AIC comparisons flatter themselves accordingly.
